<a href="https://colab.research.google.com/github/essanchristian-maker/DI-Bootcamp/blob/master/Week7_day2_Exercises_XP_VDB_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Vector Databases and RAG
Use this guided notebook and fill each TODO before running cells.

## What you'll learn
- Vector search strategies (KNN, ANN) and evaluation.
- Vector database utility (similarity search, RAG).
- Differences between vector DBs, libraries, and plugins.
- Best practices for vector store usage and performance.
- How LMs use context; embedding generation and storage.
- Querying vector stores and applying LMs for QA with retrieved context.

## What you'll build
A functional RAG pipeline with FAISS and ChromaDB, plus QA over retrieved context using a Hugging Face model.

## 0. Setup
Run the install cell once. If your platform needs system deps (e.g., libomp for FAISS), follow instructions in comments.

In [1]:
%pip install --upgrade pip
# Installation forcée de Pydantic v1 pour la compatibilité avec ChromaDB 0.3.21
%pip install "pydantic<2.0" "faiss-cpu>=1.7.4" "chromadb==0.3.21" sentence-transformers transformers pandas requests pydantic-settings
import faiss
print(f"FAISS version: {faiss.__version__}")

INFO: pip is looking at multiple versions of pydantic-settings to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of pydantic-settings to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
ERROR: Failed to build 'pyyaml' when getting requirements to build wheel
FAISS version: 1.14.3


In [2]:
import os
import json
import numpy as np
import faiss

# Fix pour l'import de BaseSettings dans les anciennes versions de ChromaDB/Pydantic
try:
    from pydantic import BaseSettings
except ImportError:
    from pydantic.v1 import BaseSettings

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer, InputExample
from transformers import pipeline
from IPython.display import display

os.makedirs("cache", exist_ok=True)
print("✅ Environnement et imports validés avec patch de compatibilité.")

✅ Environnement et imports validés avec patch de compatibilité.


In [19]:
import requests
import pandas as pd

# URL alternative fonctionnelle (version brute de GitHub)
url = "https://raw.githubusercontent.com/kotartemiy/newscatcher/master/data/labelled_newscatcher_dataset.csv"
try:
    response = requests.get(url, timeout=10)
    if response.status_code == 200:
        with open("labelled_newscatcher_dataset.csv", "wb") as f:
            f.write(response.content)
        print("✅ Dataset téléchargé avec succès !")
    else:
        # Backup: création de données fictives pour permettre la réussite des exercices si le réseau échoue
        print("⚠️ URL indisponible. Création d'un dataset de secours...")
        data = {'title': ['NASA space mission', 'New tech trends', 'Animal rescue'], 'topic': ['SCIENCE', 'TECH', 'HEALTH']}
        pd.DataFrame(data).to_csv('labelled_newscatcher_dataset.csv', sep=';', index=False)
except Exception as e:
    print(f"❌ Erreur réseau : {e}")

⚠️ URL indisponible. Création d'un dataset de secours...


## 🌟 Exercice 1 · Chargement et préparation des données

In [18]:
import pandas as pd

try:
    # Chargement avec détection du séparateur
    pdf = pd.read_csv("labelled_newscatcher_dataset.csv", sep=";")
    if pdf.shape[1] <= 1:
        pdf = pd.read_csv("labelled_newscatcher_dataset.csv", sep=",")

    pdf["id"] = range(len(pdf))
    pdf_subset = pdf.head(1000).copy()
    pdf_to_index = pdf_subset

    print(f"✅ Données chargées : {len(pdf_subset)} lignes.")
    display(pdf_subset.head())
except Exception as e:
    print(f"❌ Erreur lors de la préparation : {e}")

✅ Données chargées : 3 lignes.


,title,topic,id
0,NASA space mission,SCIENCE,0
1,New tech trends,TECH,1
2,Animal rescue,HEALTH,2


## 🌟 Exercise 2 · Vectorization with Sentence Transformers

In [10]:
from sentence_transformers import InputExample
import pandas as pd

# Tentative de récupération des données ou création d'un fallback
try:
    pdf = pd.read_csv("labelled_newscatcher_dataset.csv", sep=";")
    if pdf.shape[1] <= 1:
        pdf = pd.read_csv("labelled_newscatcher_dataset.csv", sep=",")
    pdf["id"] = range(len(pdf))
    pdf_subset = pdf.head(1000).copy()
except Exception:
    # Dataset de secours si le fichier est manquant ou invalide
    data = {'title': ['NASA space mission', 'New tech trends', 'Animal rescue'], 'topic': ['SCIENCE', 'TECH', 'HEALTH'], 'id': [0, 1, 2]}
    pdf_subset = pd.DataFrame(data)

def example_create_fn(idx, text):
    return InputExample(guid=str(idx), texts=[str(text)], label=0.0)

# Exercice 2: Création des exemples d'entraînement
faiss_train_examples = [
    example_create_fn(row["id"], row["title"])
    for _, row in pdf_subset.iterrows()
]

print(f"✅ Nombre d'exemples créés : {len(faiss_train_examples)}")

✅ Nombre d'exemples créés : 3


In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
titles_list = pdf_subset['title'].tolist()
faiss_title_embedding = model.encode(titles_list, convert_to_numpy=True, show_progress_bar=True)
len(faiss_title_embedding), len(faiss_title_embedding[0])


## 🌟 Exercise 3 · FAISS indexing and search

In [ ]:
pdf_to_index = pdf_subset
id_index = pdf_to_index['id'].to_numpy().astype(np.int64)
content_encoded_normalized = faiss_title_embedding.astype('float32')
faiss.normalize_L2(content_encoded_normalized)
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(content_encoded_normalized.shape[1]))
index_content.add_with_ids(content_encoded_normalized, id_index)
index_content.ntotal


In [16]:
import pandas as pd
import faiss
import numpy as np

if 'model' not in globals():
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('all-MiniLM-L6-v2')

if 'index_content' not in globals():
    titles_list = pdf_subset['title'].tolist()
    embeddings = model.encode(titles_list, convert_to_numpy=True)
    faiss.normalize_L2(embeddings)
    id_index = pdf_subset['id'].to_numpy().astype(np.int64)
    index_content = faiss.IndexIDMap(faiss.IndexFlatIP(embeddings.shape[1]))
    index_content.add_with_ids(embeddings.astype('float32'), id_index)

def search_content(query: str, pdf_to_index_df: pd.DataFrame, k: int = 3):
    query_vector = model.encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(query_vector)
    sims, ids = index_content.search(query_vector, k)
    results = pdf_to_index_df[pdf_to_index_df['id'].isin(ids[0])].copy()
    results['similarities'] = sims[0]
    return results.sort_values(by='similarities', ascending=False)

try:
    res = search_content('tech', pdf_subset, k=2)
    display(res)
except Exception as e:
    print(f'Erreur lors de la recherche : {e}')

,title,topic,id,similarities
0,NASA space mission,SCIENCE,0,0.593703
1,New tech trends,TECH,1,0.249398


## 🌟 Exercise 4 · ChromaDB collection and querying

In [17]:
try:
    chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
    collection_name = 'my_news_collection'
    if any(c.name == collection_name for c in chroma_client.list_collections()):
        chroma_client.delete_collection(name=collection_name)
    collection = chroma_client.create_collection(name=collection_name)
    collection.add(
        documents=pdf_subset['title'].tolist(),
        metadatas=[{'topic': t} for t in pdf_subset['topic'].tolist()],
        ids=[str(i) for i in pdf_subset['id'].tolist()]
    )
    print(f"✅ Collection {collection_name} créée et indexée.")
except Exception as e:
    print(f"Erreur ChromaDB : {e}")

ERROR:chromadb.telemetry.posthog:Failed to send telemetry event client_start: capture() takes 1 positional argument but 3 were given


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

ERROR:chromadb.telemetry.posthog:Failed to send telemetry event collection_add: capture() takes 1 positional argument but 3 were given


✅ Collection my_news_collection créée et indexée.


## 🌟 Exercise 5 · Question answering with a Hugging Face model

In [18]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import time

model_id = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model_qa = AutoModelForSeq2SeqLM.from_pretrained(model_id)

question = "What mission is NASA performing?"

# Sécurité : Attendre que l'index ChromaDB soit prêt si nécessaire
try:
    # Vérification du nombre d'éléments
    count = collection.count()
    if count == 0:
        print("⚠️ La collection est vide. Ré-indexation...")
        collection.add(
            documents=pdf_subset['title'].tolist(),
            metadatas=[{'topic': t} for t in pdf_subset['topic'].tolist()],
            ids=[str(i) for i in pdf_subset['id'].tolist()]
        )
        time.sleep(1)

    # Récupération du contexte avec gestion d'attente pour l'index
    results_chroma = collection.query(query_texts=[question], n_results=1)
    context = results_chroma["documents"][0][0]

    # Construction du prompt
    prompt = f"Context: {context}\nQuestion: {question}\nAnswer:"

    # Génération
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model_qa.generate(**inputs, max_length=50)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(f"Question : {question}")
    print(f"Contexte trouvé : {context}")
    print(f"Réponse générée : {response}")

except Exception as e:
    print(f"Erreur finale : {e}")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Question : What mission is NASA performing?
Contexte trouvé : NASA space mission
Réponse générée : space mission
